In [17]:
import requests
import aiohttp
import asyncio
from collections import deque
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from robotexclusionrulesparser import RobotExclusionRulesParser

class SimpleCrawler:
        def __init__(self, base_url, user_agent="LightningSearchBot/1.0"):
            self.base_url = base_url
            self.nloc = urlparse(self.base_url).netloc # netloc of the url
            self.user_agent = user_agent
            self.visited = set()
            self.rp = RobotExclusionRulesParser()
            self._load_robots_txt()

        def _load_robots_txt(self):
            robots_url = urljoin(self.base_url, "/robots.txt")
            try:
                    response = requests.get(robots_url, timeout=5)
                    self.rp.parse(response.text)
            except Exception as e:
                    print(f"Could not load robots.txt: {e}")
                   
                    
        async def fetch_and_parse(self, url, session):
            async with session.get(url) as response:
                if(response.status != 200):
                    print(f"Failed to retrieve {url} with status {response.status}")
                    return []
                
                html = await response.text()

                # Parse the HTML with BeautifulSoup
                # Using 'lxml' is recommended for performance over 'html.parser'
                print(type(html))
                soup = BeautifulSoup(html, 'lxml')

                # Now you can use standard BeautifulSoup methods
                out = []
                for link in soup.find_all('a',href=True):
                    next_url = urljoin(url, link['href'])
                    
                    # if url is in the same domain and hasnt been visited yet 
                    if next_url not in self.visited and urlparse(next_url).netloc == self.nloc:
                        out.append(next_url)
                        self.visited.add(next_url)
                # can return empty
                return out

        async def crawl(self, start_url, depth):
            # basically a binary tree, level-order traversal
                    urls = [start_url]
                    self.visited.add(start_url)
                    while(len(urls) and depth): # loop until either no urls can be looped into or max depth reached
                        async with aiohttp.ClientSession() as session:
                            traverse = [self.fetch_and_parse(url,session) for url in urls]
                            urls = await asyncio.gather(*traverse)
                            print(urls)
                        
                
                    # 2. Set Custom User-Agent
                    # headers = {'User-Agent': self.user_agent}
                    # response = requests.get(current_url, headers=headers, timeout=10)

                    # if response.status_code == 200:
                    #     #soup = BeautifulSoup(response.text, 'html.parser')
                    #     print(f"Depth: {depth} Crawling: {current_url}")

                    #     # 3. Extract and add internal links to the queue
                    #     for link in soup.find_all('a', href=True):
                    #         next_url = urljoin(current_url, link['href'])

                    #         # Ensure it is an internal link and not visited
                    #         if urlparse(next_url).netloc == urlparse(self.base_url).netloc:
                    #             if next_url not in self.visited:

async def main():
    crawler = SimpleCrawler("https://www.britannica.com/")
    await crawler.crawl("https://www.britannica.com/",5)
    
await main()


<class 'str'>
[['https://www.britannica.com/History-Society', 'https://www.britannica.com/Science-Tech', 'https://www.britannica.com/Biographies', 'https://www.britannica.com/Animals-Nature', 'https://www.britannica.com/Geography-Travel', 'https://www.britannica.com/Arts-Culture', 'https://www.britannica.com/procon', 'https://www.britannica.com/money', 'https://www.britannica.com/quiz/browse', 'https://www.britannica.com/videos', 'https://www.britannica.com/on-this-day', 'https://www.britannica.com/one-good-fact', 'https://www.britannica.com/dictionary', 'https://www.britannica.com/new-articles', 'https://www.britannica.com/browse/Lifestyles-Social-Issues', 'https://www.britannica.com/browse/Philosophy-Religion', 'https://www.britannica.com/browse/Politics-Law-Government', 'https://www.britannica.com/browse/World-History', 'https://www.britannica.com/browse/Health-Medicine', 'https://www.britannica.com/browse/Science', 'https://www.britannica.com/browse/Technology', 'https://www.britan

TypeError: Constructor parameter should be str